In [ ]:
from functools import partial
from typing import Dict

import optax
import tensorflow
import tensorflow_datasets
from blocks import ResNet18
from flax import linen as nn
from flax.training import early_stopping, train_state
from jax import jit, numpy, random, value_and_grad
from matplotlib import pyplot
from tqdm import tqdm
from utils import search_folder, slurm_mode

FOLDER_DATASETS = search_folder("pisco_euclid").parent
BAR = slurm_mode()


def loss_optimiser(train_losses, val_losses, steps):
    epochs = numpy.arange(1, len(train_losses) + 1)

    _, ax1 = pyplot.subplots(figsize=(6, 5))

    ax1.set_xlabel("steps")
    ax1.set_ylabel("Loss")
    ax1.plot(epochs, train_losses, "b-", label="Train")
    ax1.plot(epochs, val_losses, "g-", label="Val")
    ax1.tick_params(axis="y", labelcolor="black")
    ax1.legend(loc="lower left")

    if steps is not None:
        ax2 = ax1.twinx()
        ax2.plot(epochs, steps, "r--", label="step value")
        ax2.tick_params(axis="y", labelcolor="r")
        ax2.legend(loc="upper right")

    pyplot.title("Training & Validation Losses ")
    pyplot.tight_layout()

In [ ]:
VERSION = "0.0.0"
BATCH = 64
STEPS = 10

In [ ]:
mean_fn = partial(numpy.mean, axis=(1, 2))
ResNet_18 = partial(ResNet18)


class ResNet_MuScalar(nn.Module):
    @nn.compact
    def __call__(self, x):
        # encoder
        x = ResNet_18()(x)

        # mean of filters
        x = mean_fn(x)

        # outputs
        x = nn.Dense(2, name="Dense")(x)
        return x

In [ ]:
def rescale(x, old_min, old_max, new_min, new_max):
    old_size = old_max - old_min
    new_size = new_max - new_min
    scale_factor = new_size / old_size
    return new_min + (x - old_min) * scale_factor


BOUNDS_NORMA = numpy.array([1e-5, 1 - 1e-5])


def normalised_preds(x, old_min, old_max):
    return rescale(
        x, old_min=old_min, old_max=old_max, new_min=BOUNDS_NORMA[0], new_max=BOUNDS_NORMA[1]
    )


def range_compression(img, std_images):
    return tensorflow.clip_by_value(
        tensorflow.math.asinh(
            tensorflow.expand_dims(img, axis=-1) / tensorflow.constant(std_images) / 3.0
        ),
        -10,
        10,
    )


def preprocessing_ellips(example, std_images, old_min, old_max):
    return range_compression(example["image"], std_images), normalised_preds(
        example["avg_obs_e"], old_min, old_max
    )


std_images = numpy.float16(0.14)
min = numpy.float16(-0.7)
max = numpy.float16(0.7)
PROCESS_FUNCTION = partial(preprocessing_ellips, std_images=std_images, old_min=min, old_max=max)

In [ ]:
class TrainState(train_state.TrainState):
    batch_stats: Dict


def create_train_state():
    # take the architecture
    model = ResNet_MuScalar()

    # initialisation of the JAX model
    variables = model.init(random.key(seed=22), numpy.ones([1, 352, 352, 1]))

    # create optimizer/scheduler
    init_value = 0.001
    steps_and_lrs = [(7500, 0.5), (15000, 0.5), (20000, 0.2)]

    learning_rate = optax.schedules.piecewise_interpolate_schedule(
        interpolate_type="cosine",
        init_value=init_value,
        boundaries_and_scales={step: lr for step, lr in steps_and_lrs},
    )

    # Stochastic Gradient Descent
    optimiser = optax.adamw(learning_rate=learning_rate, weight_decay=1e-3)

    # JAX style for training loop
    state = TrainState.create(
        apply_fn=model.apply,
        params=variables["params"],
        batch_stats=variables["batch_stats"],
        tx=optimiser,
    )

    return learning_rate, state


learning_rate, state = create_train_state()

In [ ]:
# scalar case
@jit
def update_model_scalar(state, x, y):
    def loss_fn_multi(params, batch_stats):
        variables = {"params": params, "batch_stats": batch_stats}
        preds = state.apply_fn(variables, x)
        loss_val = numpy.mean(numpy.sum((preds - y) ** 2, axis=-1))
        return loss_val

    gradient_fn = value_and_grad(loss_fn_multi)
    loss, grads = gradient_fn(state.params, state.batch_stats)
    new_state = state.apply_gradients(grads=grads)
    return new_state, loss


@jit
def eval_function_scalar(state, x, y):
    variables = {"params": state.params, "batch_stats": state.batch_stats}
    preds = state.apply_fn(variables, x)
    return numpy.mean(numpy.sum((preds - y) ** 2, axis=-1))

In [ ]:
train_loader = (
    tensorflow_datasets.load(
        f"pisco_euclid:{VERSION}",
        split="dataset[:90%]",
        data_dir=FOLDER_DATASETS,
        shuffle_files=True,
    )
    .repeat()  # to not stop after the network has seen the entire dataset
    .shuffle(40000, reshuffle_each_iteration=True)
    .map(PROCESS_FUNCTION)  # wanted normalisation
    .batch(BATCH, drop_remainder=True)  # size of batch and drop if the last one is not complete
    .prefetch(-1)  # pre-load next batch while running
    .as_numpy_iterator()
)

val_loader = (
    tensorflow_datasets.load(
        f"pisco_euclid:{VERSION}",
        split="dataset[90%:95%]",
        data_dir=FOLDER_DATASETS,
        shuffle_files=False,
    )
    .repeat()
    .map(PROCESS_FUNCTION)
    .batch(BATCH, drop_remainder=True)
    .prefetch(-1)
    .as_numpy_iterator()
)

In [ ]:
early_stop = early_stopping.EarlyStopping(min_delta=1e-10, patience=100)

list_train_loss, list_val_loss, list_steps = [], [], []

for step in tqdm(range(STEPS), disable=BAR):
    batch = next(train_loader)

    state, loss = update_model_scalar(state, x=numpy.nan_to_num(batch[0]), y=numpy.array(batch[1]))
    batch = next(val_loader)
    val_loss = eval_function_scalar(state, x=numpy.nan_to_num(batch[0]), y=numpy.array(batch[1]))

    list_steps.append(learning_rate(step))
    list_train_loss.append(loss)
    list_val_loss.append(val_loss)

    early_stop = early_stop.update(metric=val_loss)
    if early_stop.should_stop:
        print(f"EARLY STOPPING at steps {step + 1}")
        break

In [ ]:
loss_optimiser(list_train_loss, list_val_loss, steps=list_steps)

In [ ]:
# from jax_resnet import pretrained_resnet, slice_variables, Sequential

# ## Encoder for pretrained model

# class EncoderPool_TL(nn.Module):

# pre_layer: Sequence[int]

# layer: Sequence[int]

# kernel_size: Tuple[int, int]

# pad_width: Union[Tuple[int, int]]


# @nn.compact

# def __call__(self, x,psf=None,train=True):


# resnet_depth = 18

# #print(x.shape)

# #x = self.preprocess(x)

# # Adjust input channels to match ResNet input (3 channels)

# x = nn.Conv(features=3, kernel_size=(1, 1))(x)

# #print(x.shape)

# # Downsample to 224x224 to match ResNet input size

# x = jax.image.resize(x, (x.shape[0], 224, 224, 3), method='bilinear')

# #print(x.shape)

# # Pre-trained ResNet from jax-resnet

# resnet_model, params = pretrained_resnet(resnet_depth)

# model = resnet_model()

# start, end = 0, len(model.layers) - 2

# backbone = Sequential(model.layers[start:end])

# x = backbone(x)

# ## Dense layer for adjusting the pretrained model to our case

# x = nn.Dense(512)(x)

# x = ACTIVATION(x)

# x = nn.Dropout(rate=0.2)(x, deterministic=False)

# x = nn.Dense(256)(x)

# x = ACTIVATION(x)

# x = nn.Dropout(rate=0.2)(x, deterministic=False)

# x = nn.Dense(128)(x)

# x = ACTIVATION(x)

# x = nn.Dropout(rate=0.2)(x, deterministic=False)

# x = nn.Dense(64)(x)

# x = ACTIVATION(x)

# x = nn.Dropout(rate=0.2)(x, deterministic=False)

# #print(x.shape)

# return x
